In [1]:

# Import libraries for comparing clustering algorithms

import numpy as np
import pandas as pd

from sklearn.metrics import silhouette_score

# Load processed customer features and metadata
X_processed = pd.read_csv("../data/processed/customer_features.csv")
metadata = pd.read_csv("../data/processed/customer_metadata.csv")

# Check the data dimensions
print("Processed features shape:", X_processed.shape)
print("Customer metadata shape:", metadata.shape)

# Preview the metadata
display(metadata.head())


Processed features shape: (7043, 45)
Customer metadata shape: (7043, 2)


,customerID,Churn
0,7590-VHVEG,No
1,5575-GNVDE,No
2,3668-QPYBK,Yes
3,7795-CFOCW,No
4,9237-HQITU,Yes


In [2]:

# Import the three clustering algorithms
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN

# 1. K-Means clustering
kmeans_model = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)
kmeans_labels = kmeans_model.fit_predict(X_processed)

# 2. Hierarchical clustering
hierarchical_model = AgglomerativeClustering(
    n_clusters=3,
    linkage="ward"
)
hierarchical_labels = hierarchical_model.fit_predict(X_processed)

# 3. DBSCAN clustering
dbscan_model = DBSCAN(
    eps=2.5,
    min_samples=10,
    metric="euclidean",
    n_jobs=-1
)
dbscan_labels = dbscan_model.fit_predict(X_processed)

print("All three algorithms completed.")
print("K-Means clusters:", np.unique(kmeans_labels))
print("Hierarchical clusters:", np.unique(hierarchical_labels))
print("DBSCAN labels:", np.unique(dbscan_labels))


All three algorithms completed.
K-Means clusters: [0 1 2]
Hierarchical clusters: [0 1 2]
DBSCAN labels: [-1  0  1  2  3]


In [3]:

# Store cluster assignments for each algorithm
cluster_assignments = {
    "K-Means": kmeans_labels,
    "Hierarchical": hierarchical_labels,
    "DBSCAN": dbscan_labels
}

# Compare the number and size of clusters
comparison_rows = []

for algorithm, labels in cluster_assignments.items():

    unique_labels = np.unique(labels)

    # DBSCAN uses -1 to represent noise, not a genuine cluster
    cluster_ids = unique_labels[unique_labels != -1]
    noise_count = int(np.sum(labels == -1))

    comparison_rows.append({
        "Algorithm": algorithm,
        "Number of Clusters": len(cluster_ids),
        "Noise Customers": noise_count,
        "Noise Percentage": round(noise_count / len(labels) * 100, 2),
        "Smallest Cluster": min(
            np.sum(labels == cluster_id)
            for cluster_id in cluster_ids
        ),
        "Largest Cluster": max(
            np.sum(labels == cluster_id)
            for cluster_id in cluster_ids
        )
    })

cluster_size_comparison = pd.DataFrame(comparison_rows)

display(cluster_size_comparison)



,Algorithm,Number of Clusters,Noise Customers,Noise Percentage,Smallest Cluster,Largest Cluster
0,K-Means,3,0,0.00,1526,3189
1,Hierarchical,3,0,0.00,1526,3462
2,DBSCAN,4,108,1.53,52,4403


In [4]:

# Use the same sample of customers for a fairer comparison
sample_size = 2000
rng = np.random.RandomState(42)

sample_indices = rng.choice(
    len(X_processed),
    size=sample_size,
    replace=False
)

X_sample = X_processed.iloc[sample_indices]

kmeans_sample_labels = kmeans_labels[sample_indices]
hierarchical_sample_labels = hierarchical_labels[sample_indices]
dbscan_sample_labels = dbscan_labels[sample_indices]

# Calculate silhouette scores
# DBSCAN's noise points are excluded from its score
silhouette_results = []

for algorithm, labels in [
    ("K-Means", kmeans_sample_labels),
    ("Hierarchical", hierarchical_sample_labels)
]:
    score = silhouette_score(X_sample, labels)

    silhouette_results.append({
        "Algorithm": algorithm,
        "Silhouette Score": round(score, 4),
        "Evaluation": "All sampled customers"
    })

# Exclude DBSCAN noise points
dbscan_non_noise_mask = dbscan_sample_labels != -1
X_dbscan_sample = X_sample.iloc[
    np.where(dbscan_non_noise_mask)[0]
]
dbscan_non_noise_labels = dbscan_sample_labels[
    dbscan_non_noise_mask
]

# A silhouette score requires at least two clusters
if len(np.unique(dbscan_non_noise_labels)) >= 2:
    dbscan_score = silhouette_score(
        X_dbscan_sample,
        dbscan_non_noise_labels
    )

    silhouette_results.append({
        "Algorithm": "DBSCAN",
        "Silhouette Score": round(dbscan_score, 4),
        "Evaluation": (
            f"Non-noise customers only "
            f"({len(dbscan_non_noise_labels)} sampled)"
        )
    })

silhouette_comparison = pd.DataFrame(silhouette_results)
display(silhouette_comparison)


,Algorithm,Silhouette Score,Evaluation
0,K-Means,0.2383,All sampled customers
1,Hierarchical,0.1996,All sampled customers
2,DBSCAN,0.1736,Non-noise customers only (1977 sampled)


In [5]:

# Combine customer churn labels with each algorithm's assignments
business_results = metadata[["Churn"]].copy()

business_results["K-Means"] = kmeans_labels
business_results["Hierarchical"] = hierarchical_labels
business_results["DBSCAN"] = dbscan_labels

# Calculate observed churn rates by cluster
churn_rate_tables = {}

for algorithm in ["K-Means", "Hierarchical", "DBSCAN"]:
    churn_table = (
        business_results
        .groupby(algorithm)
        .agg(
            Customers=("Churn", "size"),
            Churned=("Churn", lambda values: (values == "Yes").sum())
        )
    )

    churn_table["Churn Rate (%)"] = (
        churn_table["Churned"] / churn_table["Customers"] * 100
    ).round(2)

    churn_rate_tables[algorithm] = churn_table

    print(f"\n{algorithm}: observed churn by cluster")
    display(churn_table)



K-Means: observed churn by cluster


,Customers,Churned,Churn Rate (%)
K-Means,,,
0,1526,113,7.40
1,3189,1404,44.03
2,2328,352,15.12



Hierarchical: observed churn by cluster


,Customers,Churned,Churn Rate (%)
Hierarchical,,,
0,3462,1475,42.61
1,1526,113,7.40
2,2055,281,13.67



DBSCAN: observed churn by cluster


,Customers,Churned,Churn Rate (%)
DBSCAN,,,
-1,108,11,10.19
0,4403,1282,29.12
1,1474,108,7.33
2,1006,463,46.02
3,52,5,9.62


In [6]:

# Build a final summary of the three clustering algorithms

final_comparison = pd.DataFrame([
    {
        "Algorithm": "K-Means",
        "Clusters": 3,
        "Noise Customers": 0,
        "Noise (%)": 0.00,
        "Silhouette Score": 0.2383,
        "Lowest Churn (%)": 7.40,
        "Highest Churn (%)": 44.03
    },
    {
        "Algorithm": "Hierarchical",
        "Clusters": 3,
        "Noise Customers": 0,
        "Noise (%)": 0.00,
        "Silhouette Score": 0.1996,
        "Lowest Churn (%)": 7.40,
        "Highest Churn (%)": 42.61
    },
    {
        "Algorithm": "DBSCAN",
        "Clusters": 4,
        "Noise Customers": 108,
        "Noise (%)": 1.53,
        "Silhouette Score": 0.1736,
        "Lowest Churn (%)": 7.33,
        "Highest Churn (%)": 46.02
    }
])

display(final_comparison)


,Algorithm,Clusters,Noise Customers,Noise (%),Silhouette Score,Lowest Churn (%),Highest Churn (%)
0,K-Means,3,0,0.00,0.2383,7.40,44.03
1,Hierarchical,3,0,0.00,0.1996,7.40,42.61
2,DBSCAN,4,108,1.53,0.1736,7.33,46.02


# Clustering Algorithm Comparison — Final Summary

## 1. Objective

The objective was to compare K-Means, hierarchical clustering, and DBSCAN for customer segmentation using the IBM Telco Customer Churn dataset. The comparison considered cluster structure, silhouette scores, noise detection, and differences in observed customer churn rates.

## 2. Methodology

All three algorithms used the same 45 processed customer features. Customer IDs and churn labels were excluded from the clustering inputs. Churn labels were used afterward to interpret the resulting segments.

- **K-Means:** 3 clusters, `random_state=42`, `n_init=10`.
- **Hierarchical clustering:** 3 clusters using Ward linkage and Euclidean distance.
- **DBSCAN:** `eps=2.5`, `min_samples=10`, Euclidean distance.

The silhouette scores for K-Means and hierarchical clustering were calculated using the same sample of 2,000 customers. DBSCAN's score was calculated on the non-noise customers in that sample, so its score is not directly comparable on exactly the same observations.

## 3. Results

| Metric | K-Means | Hierarchical | DBSCAN |
|---|---:|---:|---:|
| Number of clusters | 3 | 3 | 4 |
| Noise customers | 0 | 0 | 108 |
| Noise percentage | 0.00% | 0.00% | 1.53% |
| Silhouette score | 0.2383 | 0.1996 | 0.1736 |
| Lowest observed churn rate | 7.40% | 7.40% | 7.33% |
| Highest observed churn rate | 44.03% | 42.61% | 46.02% |

## 4. Key Findings

### K-Means
K-Means achieved the highest silhouette score among the tested models. Its three clusters showed distinct observed churn rates of 7.40%, 44.03%, and 15.12%. This makes it a promising baseline for producing actionable customer segments.

### Hierarchical Clustering
Hierarchical clustering produced three groups with observed churn rates of 42.61%, 7.40%, and 13.67%. It revealed a similar high-churn and low-churn pattern to K-Means, although its silhouette score was lower.

### DBSCAN
DBSCAN identified four clusters and 108 noise customers. Its highest-churn cluster had an observed churn rate of 46.02%, while its lowest-churn cluster had a rate of 7.33%. However, the cluster sizes were uneven, ranging from 52 to 4,403 customers. Its silhouette score was calculated on non-noise customers only.

## 5. Business Interpretation

The results suggest that customer segments differ meaningfully in their observed churn rates. The high-churn segments identified by all three algorithms may be useful starting points for investigating retention opportunities.

However, clustering does not explain why customers churn, and a high observed churn rate does not establish that any particular feature causes churn. Further analysis would be required to identify actionable drivers and test retention strategies.

## 6. Limitations

- All three silhouette scores are relatively modest, indicating limited cluster separation.
- DBSCAN's silhouette score uses a different subset because noise points were excluded.
- Results depend on preprocessing, feature scaling, distance calculations, and model parameters.
- Clusters are descriptive groups, not churn predictions.
- The observed churn differences do not establish causation.
- No independent validation or business-impact experiment has yet been conducted.

## 7. Final Recommendation

**K-Means is the recommended baseline for this project.** It achieved the highest silhouette score among the tested configurations, produced three reasonably substantial customer segments, and revealed clear differences in observed churn rates.

Hierarchical clustering is a useful comparison because it revealed a similar segmentation pattern. DBSCAN offers a different perspective by identifying dense groups and unassigned observations, but its uneven cluster sizes make it less straightforward as the primary segmentation method here.

This recommendation is provisional: the silhouette scores are modest, and additional validation would be needed before deploying these segments in a real business setting.

## Conclusion

This project compared three unsupervised clustering algorithms to segment telecom customers and interpret the resulting groups using observed churn behavior. K-Means provided the strongest overall balance of cluster separation and interpretability among the tested configurations. The project also demonstrates why model evaluation should combine quantitative metrics with business interpretation rather than rely on a single score.